In [41]:
import os
import time
import requests
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

BASE_URL = "http://localhost:3000"

LOGIN_EMAIL    = "mathews0912.camargo@hotmail.com"
LOGIN_PASSWORD = "Teste@123"

_auth = requests.post(
    f"{BASE_URL}/auth/login",
    json={"email": LOGIN_EMAIL, "password": LOGIN_PASSWORD},
    headers={"Content-Type": "application/json"},
)
_auth.raise_for_status()

HEADERS = {
    "Authorization": f"Bearer {_auth.json()['token']}",
    "Content-Type": "application/json",
}

print("Autenticado com sucesso.")

ConnectionError: HTTPConnectionPool(host='localhost', port=3000): Max retries exceeded with url: /auth/login (Caused by NewConnectionError("HTTPConnection(host='localhost', port=3000): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
# Adicione os CNPJs aqui, com ou sem formatação (ex: "00.000.000/0001-00" ou "00000000000100")
cnpj_list = [
    "58574093000148", "49805911000185", "44970334000163", "08932053000152",
    "31129072000167", "63637446000141", "65532295000192", "66339178000170",
    "64927748000117", "49098175000172", "63690848000100", "65295353000101",
    "57654948000188", "64588475000123", "43502592000152", "24500144000175",
    "32401996000133", "62205035000114", "65711607000125", "52657030000179",
    "45841567000129", "33952858000105", "47921676000181", "65301484000154",
    "44445409000197", "46291474000130", "50256393000177", "30506733000163",
    "61362479000109", "60818617000140", "35384964000165", "55296233000175",
    "64009098000120", "59487700000103", "20476755000174", "66086058000108",
    "62286579000158", "37354488000165", "57672690000142", "47106843000130",
    "27087435000171", "28283710000195", "53089478000104", "05413827000140",
    "53006830000192", "46212595000140", "33722328000170", "44818746000182",
    "64545017000107", "54124622000150", "65537929000108", "66133698000121",
    "32716533000160", "44733411000161", "55054881000115", "05953147000110",
    "46960988000131", "38141386000124", "48090589000192", "49281714000104",
    "64222535000190", "64832195000119", "63912728000100", "38457813000188",
    "63782931000109", "30074807000130", "36470973000131", "63383233000130",
    "19751209000115", "64547065000134", "48608736000173", "48326506000111",
    "47910333000111", "61004658000166", "60724151000114", "65429422000122",
    "54462554000130", "36289293000116", "63683912000125", "44921027000192",
    "62128865000195", "65279312000121", "25308848000103", "45850449000187",
    "38141386000124", "55135691000122", "64654658000108", "40046669000139",
    "59079205000157", "66006219000106", "57626965000101", "60152155000175",
    "36426523000141", "17688353000129", "43069305000162", "24990599000116",
    "64568453000100", "40278450000165", "66111008000133", "13322545000166",
    "61959853000140", "60365057000116", "30135012000194", "24177059000118",
    "14392608000113", "51293954000170", "43251482000165", "61523033000100",
    "36476993000110", "34756388000177", "60378580000187", "35752782000108",
    "11588752001960", "30119607000156", "59066504000157", "41755035000118",
    "23230777000148", "62532446000114", "27001687000136", "23946883000122",
    "65667102000100", "08271937000103", "43175790000159", "65971675000123",
    "30890601000188", "55119942000185", "64754134000180", "00549593000220",
    "55247048000190", "43951881000139", "02104067000100", "25131094000169",
    "30056759000157", "05784234000190", "51319557000120", "46509464000129",
    "24509083000107", "41449608000185", "47273278000104", "26370892000106",
    "26309442000108", "28042608000106", "13597403000101", "04941638000188",
    "37537564000178", "58027534000191", "06007549000194", "62212798000192",
    "09270215000105", "44612909000176", "24023289000122", "50018862000110",
    "48048012000112", "39292053000169", "37253780000191", "37472528000173",
    "47300311000130", "50075448000142", "50751906000116", "35001910000173",
    "48487427000192", "43330278000130", "62419764000173", "64847473000101",
    "38597576000150", "51517956000104", "91362590011354", "08645725000149",
    "12612913000148", "65085325000160", "43879509000169", "43175790000159",
    "65975592000102", "41849941000181", "57393689000189", "33997174000120",
    "06375850000150", "35043203000140", "58272206000150", "29991508000180",
    "49971220000151", "42039689000108", "95868097000165", "37330654000193",
    "18914048000170", "35306509000141", "59213618000182", "35917730000136",
    "01316153000105", "62788326000182", "08844430000100", "63569432000138",
    "64285275000100", "62749954000159", "40460366000168", "58107065000110",
    "63894951000171", "37936737000120", "06032176000101", "55722629000137"
]
print(f"Total de documentos a processar: {len(cnpj_list)}")

Total de documentos a processar: 200


In [ ]:
def clean_doc(doc):
    return doc.replace(".", "").replace("/", "").replace("-", "").strip()


def get_main_address(addresses):
    if not addresses:
        return "N/D"
    for addr in addresses:
        if addr.get("main") and addr.get("active", True):
            return addr.get("address", "N/D")
    for addr in addresses:
        if addr.get("active", True):
            return addr.get("address", "N/D")
    return addresses[0].get("address", "N/D")


def fmt_phone(phones):
    if not phones:
        return "N/D"
    phone = str(phones[0].get("phone", "")).strip()
    return phone if phone else "N/D"


print("Funções auxiliares carregadas.")

Funções auxiliares carregadas.


In [ ]:
WEBHOOK     = "https://webhook.site/c1a70033-31b9-4879-a4e2-6f7ffbb7bd40"
COST_CENTER = 1


def fetch_dados_gerais(document, max_retries=15, wait_sec=2):
    body = {
        "document":    document,
        "cost_center": COST_CENTER,
        "webhook_url": WEBHOOK,
    }
    try:
        r = requests.post(f"{BASE_URL}/dados-gerais", headers=HEADERS, json=body)
        r.raise_for_status()
        order_id = r.json()["order_id"]
    except Exception as e:
        print(f"  Erro ao criar ordem [{document}]: {e}")
        return None

    for _ in range(max_retries):
        try:
            r = requests.get(f"{BASE_URL}/dados-gerais/{order_id}", headers=HEADERS)
            r.raise_for_status()
            result = r.json()
            if result.get("status") == "SUCESSO":
                return result
        except Exception as e:
            print(f"  Erro ao buscar ordem {order_id} [{document}]: {e}")
            return None
        time.sleep(wait_sec)

    print(f"  Timeout aguardando resultado para {document} (order_id={order_id})")
    return None


print("Função de consulta carregada.")

In [ ]:
def process_cnpj(cnpj, errors):
    rows = []
    doc = clean_doc(cnpj)

    print(f"  Consultando dados gerais da empresa...")
    reg = fetch_dados_gerais(doc)
    if reg is None:
        errors.append((cnpj, "Erro em /dados-gerais"))
        return rows

    dp = reg.get("data", {}).get("dados_pessoais", {})
    if not dp:
        errors.append((cnpj, "Resposta sem dados_pessoais"))
        return rows

    biz_cnpj = dp.get("document", "N/D")
    biz_name = dp.get("name", "N/D")
    biz_addr = get_main_address(dp.get("address", []))
    biz_stat = dp.get("fiscal_situation", "N/D")
    biz_type = dp.get("legal_nature", "N/D")

    partners  = reg.get("data", {}).get("sociedades", [])
    ubo_names = ", ".join(p.get("name", "") for p in partners) if partners else "N/D"

    if not partners:
        rows.append({
            "Business CNPJ":       biz_cnpj,
            "Business Name":       biz_name,
            "Business Address":    biz_addr,
            "Registration Status": biz_stat,
            "Entity Type":         biz_type,
            "List of UBOs":        "N/D",
            "UBOs Ownership (%)":  "N/D",
            "UBO Name":            "N/D",
            "UBO Tax ID":          "N/D",
            "UBO Address":         "N/D",
            "Tax Status":          "N/D",
            "Phone":               "N/D",
            "Date of Birth":       "N/D",
        })
        return rows

    for partner in partners:
        ubo_raw_doc = partner.get("document", "")
        ubo_doc     = clean_doc(ubo_raw_doc)
        ubo_name    = partner.get("name", "N/D")
        ubo_own     = partner.get("participation", partner.get("relation", "N/D"))

        rows.append({
            "Business CNPJ":       biz_cnpj,
            "Business Name":       biz_name,
            "Business Address":    biz_addr,
            "Registration Status": biz_stat,
            "Entity Type":         biz_type,
            "List of UBOs":        ubo_names,
            "UBOs Ownership (%)":  ubo_own,
            "UBO Name":            ubo_name,
            "UBO Tax ID":          ubo_doc or "N/D",
            "UBO Address":         "N/D",
            "Tax Status":          "N/D",
            "Phone":               "N/D",
            "Date of Birth":       "N/D",
        })

    return rows


print("Função de processamento carregada.")

In [ ]:
all_rows = []
errors   = []

for cnpj in cnpj_list:
    print(f"\nProcessando: {cnpj}")
    all_rows.extend(process_cnpj(cnpj, errors))

with open("erros_registro.txt", "w", encoding="utf-8") as f:
    f.write(f"Erros de consulta — {len(errors)} ocorrência(s)\n")
    f.write("=" * 50 + "\n\n")
    for doc, reason in errors:
        f.write(f"{doc} | {reason}\n")

print(f"\n--- Resumo ---")
print(f"Linhas geradas  : {len(all_rows)}")
print(f"Erros registrados: {len(errors)}")
if errors:
    print("Arquivo 'erros_registro.txt' criado com os documentos com falha.")


Processando: 58574093000148
  Consultando dados gerais da empresa...
  Erro ao criar ordem [58574093000148]: 500 Server Error: Internal Server Error for url: http://localhost:3000/dados-gerais

Processando: 49805911000185
  Consultando dados gerais da empresa...
  Erro ao criar ordem [49805911000185]: 500 Server Error: Internal Server Error for url: http://localhost:3000/dados-gerais

Processando: 44970334000163
  Consultando dados gerais da empresa...
  Erro ao criar ordem [44970334000163]: 500 Server Error: Internal Server Error for url: http://localhost:3000/dados-gerais

Processando: 08932053000152
  Consultando dados gerais da empresa...
  Erro ao criar ordem [08932053000152]: 500 Server Error: Internal Server Error for url: http://localhost:3000/dados-gerais

Processando: 31129072000167
  Consultando dados gerais da empresa...
  Erro ao criar ordem [31129072000167]: 500 Server Error: Internal Server Error for url: http://localhost:3000/dados-gerais

Processando: 63637446000141
 

In [ ]:
COLS_BIZ = [
    "Business CNPJ",
    "Business Name",
    "Business Address",
    "Registration Status",
    "Entity Type",
    "List of UBOs",
    "UBOs Ownership (%)",
]
COLS_UBO = [
    "UBO Name",
    "UBO Tax ID",
    "UBO Address",
    "Tax Status",
    "Phone",
    "Date of Birth",
]
ALL_COLS = COLS_BIZ + COLS_UBO

FILL_BIZ_MAIN = PatternFill("solid", fgColor="BDD7EE")
FILL_BIZ_SUB  = PatternFill("solid", fgColor="DDEBF7")
FILL_UBO_MAIN = PatternFill("solid", fgColor="FCE4D6")
FILL_UBO_SUB  = PatternFill("solid", fgColor="FDE9D9")

FONT_BOLD  = Font(bold=True)
ALIGN_CTR  = Alignment(horizontal="center", vertical="center", wrap_text=True)
ALIGN_LEFT = Alignment(horizontal="left",   vertical="center", wrap_text=True)

wb = Workbook()
ws = wb.active
ws.title = "Relatório UBO"

n_biz         = len(COLS_BIZ)
n_ubo         = len(COLS_UBO)
ubo_start_col = n_biz + 1

biz_end_letter = get_column_letter(n_biz)
ubo_start_letter = get_column_letter(ubo_start_col)
ubo_end_letter = get_column_letter(n_biz + n_ubo)

# Row 1 — merged group headers
ws.merge_cells(f"A1:{biz_end_letter}1")
cell_biz = ws["A1"]
cell_biz.value     = "Business CNPJ"
cell_biz.fill      = FILL_BIZ_MAIN
cell_biz.font      = FONT_BOLD
cell_biz.alignment = ALIGN_CTR

ws.merge_cells(f"{ubo_start_letter}1:{ubo_end_letter}1")
cell_ubo = ws[f"{ubo_start_letter}1"]
cell_ubo.value     = "UBO Name"
cell_ubo.fill      = FILL_UBO_MAIN
cell_ubo.font      = FONT_BOLD
cell_ubo.alignment = ALIGN_CTR

ws.row_dimensions[1].height = 28

# Row 2 — sub-column headers
for i, col_name in enumerate(ALL_COLS, start=1):
    cell = ws.cell(row=2, column=i, value=col_name)
    cell.font      = FONT_BOLD
    cell.alignment = ALIGN_CTR
    cell.fill      = FILL_BIZ_SUB if i <= n_biz else FILL_UBO_SUB

ws.row_dimensions[2].height = 36

# Data rows
for row_idx, row_data in enumerate(all_rows, start=3):
    for col_idx, col_name in enumerate(ALL_COLS, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=row_data.get(col_name, "N/D"))
        cell.alignment = ALIGN_LEFT

# Auto-fit column widths
for col_idx, col_name in enumerate(ALL_COLS, start=1):
    col_letter = get_column_letter(col_idx)
    max_len = len(col_name)
    for row in ws.iter_rows(min_row=3, max_row=max(ws.max_row, 3),
                             min_col=col_idx, max_col=col_idx):
        for cell in row:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
    ws.column_dimensions[col_letter].width = min(max_len + 4, 50)

output_file = "relatorio_ubo.xlsx"
wb.save(output_file)
print(f"Arquivo '{output_file}' gerado com sucesso.")
print(f"  Total de linhas de dados: {len(all_rows)}")

Arquivo 'relatorio_ubo.xlsx' gerado com sucesso.
  Total de linhas de dados: 0
